# 01 Data and Baselines
#
This notebook does only four things:
1. Load and merge the IEEE-CIS training files
2. Create a leakage-safe temporal split
3. Build simple readable tabular features
4. Train LightGBM and XGBoost baselines and save outputs
#
It is written for Kaggle and designed to be easy to read and continue later.

In [1]:
import gc
import json
import os
import random
import time
import warnings
from pathlib import Path
from time import perf_counter

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## Cell 1: Notebook Settings
#
Update only the paths below if your Kaggle input slug is different.

In [2]:
SEED = 42

RAW_DATA_ROOT = Path("/kaggle/input/competitions/ieee-fraud-detection")
OUTPUT_ROOT = Path("/kaggle/working/thesis_outputs")
PHASE_ROOT = OUTPUT_ROOT / "artifacts" / "phase01_data_baselines"

TARGET_COL = "isFraud"
ID_COL = "TransactionID"
TIME_COL = "TransactionDT"

TRAIN_TRANSACTION_PATH = RAW_DATA_ROOT / "train_transaction.csv"
TRAIN_IDENTITY_PATH = RAW_DATA_ROOT / "train_identity.csv"

REFRESH_DATA = False
REFRESH_MODELS = False
RUN_MODEL_SEARCH = True

TRAIN_RATIO = 0.75
VALID_RATIO = 0.10
TEST_RATIO = 0.15
OOF_WARMUP_FRACTION = 0.40
OOF_NUM_FOLDS = 3

ANALYST_BUDGETS = [100, 250, 500, 1000, 2500, 5000]
LATENCY_BENCHMARK_REPEATS = 20
LATENCY_BENCHMARK_WARMUP = 3

for folder in [
    PHASE_ROOT,
    OUTPUT_ROOT / "manifests",
]:
    folder.mkdir(parents=True, exist_ok=True)

## Cell 2: Reproducibility

In [3]:
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

## Cell 3: Small Helper Functions
#
Only a few helpers are kept so the notebook stays readable.

In [4]:
def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)

def recall_at_fpr(y_true, y_score, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    valid_idx = np.where(fpr <= target_fpr)[0]
    if len(valid_idx) == 0:
        return 0.0
    return float(tpr[valid_idx[-1]])


def recall_at_k(y_true, y_score, k):
    sorted_idx = np.argsort(-y_score)[:k]
    positives = max(int(np.sum(y_true)), 1)
    return float(np.sum(y_true[sorted_idx]) / positives)


def precision_at_k(y_true, y_score, k):
    sorted_idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[sorted_idx]))


def best_iteration_for_xgb(model):
    return model.best_iteration + 1 if model.best_iteration is not None else 0


def benchmark_latency_ms_per_row(predict_fn, data, repeats=LATENCY_BENCHMARK_REPEATS, warmup=LATENCY_BENCHMARK_WARMUP):
    n_rows = max(len(data), 1)

    for _ in range(warmup):
        _ = predict_fn(data)

    timings = []
    for _ in range(repeats):
        start = perf_counter()
        _ = predict_fn(data)
        end = perf_counter()
        timings.append((end - start) * 1000.0 / n_rows)

    return {
        "mean_ms_per_row": float(np.mean(timings)),
        "median_ms_per_row": float(np.median(timings)),
        "std_ms_per_row": float(np.std(timings)),
        "repeats": int(repeats),
        "rows_benchmarked": int(n_rows),
    }


def build_temporal_oof_folds(n_rows, warmup_fraction=OOF_WARMUP_FRACTION, num_folds=OOF_NUM_FOLDS):
    warmup_end = int(n_rows * warmup_fraction)
    warmup_end = max(warmup_end, 1)
    remaining = n_rows - warmup_end
    if remaining <= 0:
        return []

    base_fold_size = max(remaining // num_folds, 1)
    folds = []
    eval_start = warmup_end
    for fold_idx in range(num_folds):
        eval_end = eval_start + base_fold_size
        if fold_idx == num_folds - 1:
            eval_end = n_rows
        eval_end = min(eval_end, n_rows)
        if eval_start >= eval_end:
            break
        folds.append(
            {
                "fold_id": int(fold_idx + 1),
                "train_end": int(eval_start),
                "eval_start": int(eval_start),
                "eval_end": int(eval_end),
            }
        )
        eval_start = eval_end
        if eval_start >= n_rows:
            break
    return folds


def fit_lgbm_with_inner_early_stopping(X_prefix, y_prefix, categorical_columns, best_params):
    n_prefix = len(X_prefix)
    inner_eval_start = max(int(n_prefix * 0.9), 1)
    inner_eval_start = min(inner_eval_start, n_prefix - 1)
    X_inner_train = X_prefix.iloc[:inner_eval_start].copy()
    y_inner_train = y_prefix[:inner_eval_start]
    X_inner_eval = X_prefix.iloc[inner_eval_start:].copy()
    y_inner_eval = y_prefix[inner_eval_start:]

    model_for_iteration = lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        max_depth=-1,
        subsample_freq=1,
        scale_pos_weight=float((len(y_inner_train) - y_inner_train.sum()) / max(y_inner_train.sum(), 1)),
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        **best_params,
    )
    model_for_iteration.fit(
        X_inner_train,
        y_inner_train,
        eval_set=[(X_inner_eval, y_inner_eval)],
        eval_metric="average_precision",
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(200, first_metric_only=True), lgb.log_evaluation(0)],
    )
    best_iteration = int(model_for_iteration.best_iteration_)

    final_model = lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        max_depth=-1,
        subsample_freq=1,
        scale_pos_weight=float((len(y_prefix) - y_prefix.sum()) / max(y_prefix.sum(), 1)),
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        n_estimators=best_iteration,
        num_leaves=best_params["num_leaves"],
        min_child_samples=best_params["min_child_samples"],
        colsample_bytree=best_params["colsample_bytree"],
        subsample=best_params["subsample"],
        reg_lambda=best_params["reg_lambda"],
        reg_alpha=best_params["reg_alpha"],
        learning_rate=best_params["learning_rate"],
    )
    final_model.fit(
        X_prefix,
        y_prefix,
        categorical_feature=categorical_columns,
    )
    return final_model, best_iteration


def fit_xgb_with_inner_early_stopping(X_prefix, y_prefix, feature_columns, best_params):
    n_prefix = len(X_prefix)
    inner_eval_start = max(int(n_prefix * 0.9), 1)
    inner_eval_start = min(inner_eval_start, n_prefix - 1)
    X_inner_train = X_prefix.iloc[:inner_eval_start].copy()
    y_inner_train = y_prefix[:inner_eval_start]
    X_inner_eval = X_prefix.iloc[inner_eval_start:].copy()
    y_inner_eval = y_prefix[inner_eval_start:]

    dtrain_inner = xgb.DMatrix(X_inner_train, label=y_inner_train, feature_names=feature_columns)
    deval_inner = xgb.DMatrix(X_inner_eval, label=y_inner_eval, feature_names=feature_columns)

    inner_params = best_params.copy()
    inner_params["scale_pos_weight"] = float((len(y_inner_train) - y_inner_train.sum()) / max(y_inner_train.sum(), 1))
    model_for_iteration = xgb.train(
        params=inner_params,
        dtrain=dtrain_inner,
        num_boost_round=4000,
        evals=[(deval_inner, "valid")],
        early_stopping_rounds=200,
        verbose_eval=False,
    )
    best_iteration = int(best_iteration_for_xgb(model_for_iteration))

    final_params = best_params.copy()
    final_params["scale_pos_weight"] = float((len(y_prefix) - y_prefix.sum()) / max(y_prefix.sum(), 1))
    dfull = xgb.DMatrix(X_prefix, label=y_prefix, feature_names=feature_columns)
    final_model = xgb.train(
        params=final_params,
        dtrain=dfull,
        num_boost_round=best_iteration,
        evals=[],
        verbose_eval=False,
    )
    return final_model, best_iteration

## Cell 4: Load and Merge Raw Data
#
We use only Kaggle train files because they contain labels.

In [5]:
MERGED_PATH = PHASE_ROOT / "merged_train.parquet"

if MERGED_PATH.exists() and not REFRESH_DATA:
    print("Loading cached merged data")
    df = pd.read_parquet(MERGED_PATH)
else:
    print("Reading raw CSV files from Kaggle input")
    train_transaction = pd.read_csv(TRAIN_TRANSACTION_PATH)
    train_identity = pd.read_csv(TRAIN_IDENTITY_PATH)

    df = train_transaction.merge(train_identity, on=ID_COL, how="left")
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    df.to_parquet(MERGED_PATH, index=False)

print(df.shape)
df[[ID_COL, TIME_COL, TARGET_COL]].head()

Reading raw CSV files from Kaggle input
(590540, 434)


,TransactionID,TransactionDT,isFraud
0,2987000,86400,0
1,2987001,86401,0
2,2987002,86469,0
3,2987003,86499,0
4,2987004,86506,0


## Cell 5: Quick Data Checks
#
These checks are useful for the thesis and for debugging before training.

In [6]:
print("Fraud rate:", round(df[TARGET_COL].mean(), 4))
print("Time range:", int(df[TIME_COL].min()), "to", int(df[TIME_COL].max()))
print("Columns:", df.shape[1])
print("Missing cells:", int(df.isna().sum().sum()))

Fraud rate: 0.035
Time range: 86400 to 15811131
Columns: 434
Missing cells: 115523073


## Cell 6: Feature Engineering
#
Keep it simple and readable:
- time features
- log transaction amount
- count features for useful entity columns
- simple interaction keys

In [7]:
df["row_missing_count"] = df.isna().sum(axis=1).astype("int16")
df["row_missing_ratio"] = (df["row_missing_count"] / df.shape[1]).astype("float32")

df["dt_hour"] = ((df[TIME_COL] // 3600) % 24).astype("int16")
df["dt_dayofweek"] = ((df[TIME_COL] // 86400) % 7).astype("int16")
df["dt_week"] = (df[TIME_COL] // (86400 * 7)).astype("int32")

if "TransactionAmt" in df.columns:
    df["transaction_amt_log1p"] = np.log1p(df["TransactionAmt"].clip(lower=0))
    df["transaction_amt_is_zero"] = (df["TransactionAmt"] == 0).astype("int8")

if "dist1" in df.columns:
    df["dist1_log1p"] = np.log1p(df["dist1"].clip(lower=0))

if "dist2" in df.columns:
    df["dist2_log1p"] = np.log1p(df["dist2"].clip(lower=0))

if "card1" in df.columns and "addr1" in df.columns:
    df["card_addr_key"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str)

if "card1" in df.columns and "card2" in df.columns:
    df["card12_key"] = df["card1"].astype(str) + "_" + df["card2"].astype(str)

if "card1" in df.columns and "P_emaildomain" in df.columns:
    df["card_email_key"] = df["card1"].astype(str) + "_" + df["P_emaildomain"].astype(str)

if "card1" in df.columns and "DeviceType" in df.columns:
    df["card_device_key"] = df["card1"].astype(str) + "_" + df["DeviceType"].astype(str)

count_columns = [
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
    "ProductCD",
    "card_addr_key",
    "card12_key",
    "card_email_key",
    "card_device_key",
]

count_columns = [col for col in count_columns if col in df.columns]
count_columns

['card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'P_emaildomain',
 'R_emaildomain',
 'DeviceType',
 'DeviceInfo',
 'ProductCD',
 'card_addr_key',
 'card12_key',
 'card_email_key',
 'card_device_key']

## Cell 7: Temporal Split
#
This is critical. The split must respect time order.

In [8]:
assert abs(TRAIN_RATIO + VALID_RATIO + TEST_RATIO - 1.0) < 1e-9

n_rows = len(df)
train_end = int(n_rows * TRAIN_RATIO)
valid_end = train_end + int(n_rows * VALID_RATIO)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

print("Train:", train_df.shape, "fraud rate:", round(train_df[TARGET_COL].mean(), 4))
print("Valid:", valid_df.shape, "fraud rate:", round(valid_df[TARGET_COL].mean(), 4))
print("Test :", test_df.shape, "fraud rate:", round(test_df[TARGET_COL].mean(), 4))

Train: (442905, 447) fraud rate: 0.0351
Valid: (59054, 447) fraud rate: 0.0342
Test : (88581, 447) fraud rate: 0.0348


## Cell 8: Fit Count Maps On Train Only
#
This avoids leakage.

In [9]:
count_maps = {}
for col in count_columns:
    count_maps[col] = train_df[col].fillna("__MISSING__").astype(str).value_counts().to_dict()

for split_df in [train_df, valid_df, test_df]:
    for col in count_columns:
        split_df[f"{col}_count"] = (
            split_df[col]
            .fillna("__MISSING__")
            .astype(str)
            .map(count_maps[col])
            .fillna(0)
            .astype("float32")
        )

## Cell 8B: Train-Only Aggregation Features
#
These features are still readable, but they usually help both tree models a lot.

In [10]:
aggregation_groups = ["card1", "card12_key", "card_addr_key", "addr1", "P_emaildomain"]
aggregation_groups = [col for col in aggregation_groups if col in train_df.columns and "TransactionAmt" in train_df.columns]

for group_col in aggregation_groups:
    amount_stats = (
        train_df.groupby(group_col)["TransactionAmt"]
        .agg(["mean", "std"])
        .rename(columns={"mean": f"{group_col}_amt_mean", "std": f"{group_col}_amt_std"})
    )

    for split_df in [train_df, valid_df, test_df]:
        split_df[f"{group_col}_amt_mean"] = split_df[group_col].map(amount_stats[f"{group_col}_amt_mean"])
        split_df[f"{group_col}_amt_std"] = split_df[group_col].map(amount_stats[f"{group_col}_amt_std"])

        split_df[f"{group_col}_amt_mean"] = split_df[f"{group_col}_amt_mean"].fillna(-1).astype("float32")
        split_df[f"{group_col}_amt_std"] = split_df[f"{group_col}_amt_std"].fillna(-1).astype("float32")

        split_df[f"{group_col}_amt_to_mean_ratio"] = (
            split_df["TransactionAmt"] / split_df[f"{group_col}_amt_mean"].replace({0: np.nan})
        ).replace([np.inf, -np.inf], np.nan).fillna(-1).astype("float32")

## Cell 9: Fill Missing Values and Encode Categories
#
We fit encodings on train only, then apply them to valid and test.

In [11]:
drop_from_features = [TARGET_COL, ID_COL]

feature_columns = [col for col in train_df.columns if col not in drop_from_features]

categorical_columns = [col for col in feature_columns if train_df[col].dtype == "object"]
numeric_columns = [col for col in feature_columns if col not in categorical_columns]

numeric_fill_values = {}
for col in numeric_columns:
    median_value = train_df[col].median()
    if pd.isna(median_value):
        median_value = -999
    numeric_fill_values[col] = median_value

for split_df in [train_df, valid_df, test_df]:
    for col in numeric_columns:
        split_df[col] = split_df[col].fillna(numeric_fill_values[col])
    for col in categorical_columns:
        split_df[col] = split_df[col].fillna("__MISSING__").astype(str)

category_maps = {}
for col in categorical_columns:
    value_counts = train_df[col].value_counts()
    kept_values = set(value_counts[value_counts >= 20].index.tolist())
    category_maps[col] = {value: idx for idx, value in enumerate(sorted(kept_values | {"__MISSING__", "__RARE__"}))}

for split_df in [train_df, valid_df, test_df]:
    for col in categorical_columns:
        split_df[col] = split_df[col].apply(lambda x: x if x in category_maps[col] else "__RARE__")
        split_df[col] = split_df[col].map(category_maps[col]).astype("int32")

feature_columns = [col for col in train_df.columns if col not in drop_from_features]

print("Feature count:", len(feature_columns))
print("Categorical count:", len(categorical_columns))
print("Numeric count:", len(numeric_columns))

Feature count: 477
Categorical count: 35
Numeric count: 442


## Cell 10: Save Processed Splits
#
These files are the hand-off to later notebooks.

In [12]:
TRAIN_PATH = PHASE_ROOT / "train.parquet"
VALID_PATH = PHASE_ROOT / "valid.parquet"
TEST_PATH = PHASE_ROOT / "test.parquet"
FEATURE_SCHEMA_PATH = PHASE_ROOT / "feature_schema.json"
CATEGORY_MAPS_PATH = PHASE_ROOT / "category_maps.json"
SPLIT_INFO_PATH = PHASE_ROOT / "split_info.json"

train_df.to_parquet(TRAIN_PATH, index=False)
valid_df.to_parquet(VALID_PATH, index=False)
test_df.to_parquet(TEST_PATH, index=False)

save_json(
    {
        "feature_columns": feature_columns,
        "categorical_columns": categorical_columns,
        "numeric_columns": numeric_columns,
        "target_column": TARGET_COL,
        "id_column": ID_COL,
        "time_column": TIME_COL,
    },
    FEATURE_SCHEMA_PATH,
)

save_json(category_maps, CATEGORY_MAPS_PATH)
save_json(
    {
        "train_rows": len(train_df),
        "valid_rows": len(valid_df),
        "test_rows": len(test_df),
        "train_fraud_rate": float(train_df[TARGET_COL].mean()),
        "valid_fraud_rate": float(valid_df[TARGET_COL].mean()),
        "test_fraud_rate": float(test_df[TARGET_COL].mean()),
        "train_time_min": int(train_df[TIME_COL].min()),
        "train_time_max": int(train_df[TIME_COL].max()),
        "valid_time_min": int(valid_df[TIME_COL].min()),
        "valid_time_max": int(valid_df[TIME_COL].max()),
        "test_time_min": int(test_df[TIME_COL].min()),
        "test_time_max": int(test_df[TIME_COL].max()),
    },
    SPLIT_INFO_PATH,
)

## Cell 11: Build Model Matrices

In [13]:
X_train = train_df[feature_columns].copy()
y_train = train_df[TARGET_COL].astype(int).values

X_valid = valid_df[feature_columns].copy()
y_valid = valid_df[TARGET_COL].astype(int).values

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET_COL].astype(int).values

scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
print("scale_pos_weight:", round(float(scale_pos_weight), 2))

scale_pos_weight: 27.46


## Cell 12: Train LightGBM
#
This cell tries several strong candidate settings and keeps the best one.
That gives us a much fairer thesis baseline than a single fixed setup.

In [14]:
LGBM_MODEL_PATH = PHASE_ROOT / "lgbm_model.pkl"
LGBM_EVAL_PATH = PHASE_ROOT / "lgbm_eval_history.json"
LGBM_SEARCH_PATH = PHASE_ROOT / "lgbm_search_results.csv"
LGBM_PARAMS_PATH = PHASE_ROOT / "lgbm_best_params.json"

if LGBM_MODEL_PATH.exists() and not REFRESH_MODELS:
    print("Loading cached LightGBM model")
    lgbm_model = joblib.load(LGBM_MODEL_PATH)
    lgbm_eval_history = json.load(open(LGBM_EVAL_PATH, "r", encoding="utf-8"))
    best_lgbm_params = load_json(LGBM_PARAMS_PATH)
else:
    lgbm_candidates = [
        {
            "learning_rate": 0.02,
            "n_estimators": 4000,
            "num_leaves": 255,
            "min_child_samples": 40,
            "colsample_bytree": 0.7,
            "subsample": 0.8,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
        },
        {
            "learning_rate": 0.03,
            "n_estimators": 3000,
            "num_leaves": 127,
            "min_child_samples": 60,
            "colsample_bytree": 0.75,
            "subsample": 0.8,
            "reg_lambda": 2.0,
            "reg_alpha": 0.0,
        },
        {
            "learning_rate": 0.05,
            "n_estimators": 2500,
            "num_leaves": 255,
            "min_child_samples": 80,
            "colsample_bytree": 0.8,
            "subsample": 0.9,
            "reg_lambda": 3.0,
            "reg_alpha": 0.0,
        },
        {
            "learning_rate": 0.02,
            "n_estimators": 5000,
            "num_leaves": 511,
            "min_child_samples": 30,
            "colsample_bytree": 0.65,
            "subsample": 0.8,
            "reg_lambda": 1.0,
            "reg_alpha": 0.5,
        },
        {
            "learning_rate": 0.04,
            "n_estimators": 2500,
            "num_leaves": 95,
            "min_child_samples": 100,
            "colsample_bytree": 0.85,
            "subsample": 0.85,
            "reg_lambda": 4.0,
            "reg_alpha": 1.0,
        },
    ] if RUN_MODEL_SEARCH else [
        {
            "learning_rate": 0.03,
            "n_estimators": 3000,
            "num_leaves": 127,
            "min_child_samples": 60,
            "colsample_bytree": 0.75,
            "subsample": 0.8,
            "reg_lambda": 2.0,
            "reg_alpha": 0.0,
        }
    ]

    lgbm_search_rows = []
    best_lgbm_params = None
    best_lgbm_ap = -1.0

    for idx, candidate in enumerate(lgbm_candidates, start=1):
        print(f"LightGBM candidate {idx}/{len(lgbm_candidates)}")

        candidate_model = lgb.LGBMClassifier(
            objective="binary",
            boosting_type="gbdt",
            max_depth=-1,
            subsample_freq=1,
            scale_pos_weight=float(scale_pos_weight),
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
            **candidate,
        )

        candidate_model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="average_precision",
            categorical_feature=categorical_columns,
            callbacks=[lgb.early_stopping(200, first_metric_only=True), lgb.log_evaluation(0)],
        )

        candidate_pred = candidate_model.predict_proba(X_valid)[:, 1]
        candidate_ap = average_precision_score(y_valid, candidate_pred)

        lgbm_search_rows.append(
            {
                "candidate_id": idx,
                "valid_auc_pr": float(candidate_ap),
                "best_iteration": int(candidate_model.best_iteration_),
                **candidate,
            }
        )

        if candidate_ap > best_lgbm_ap:
            best_lgbm_ap = float(candidate_ap)
            best_lgbm_params = candidate.copy()

    lgbm_search_df = pd.DataFrame(lgbm_search_rows).sort_values("valid_auc_pr", ascending=False)
    lgbm_search_df.to_csv(LGBM_SEARCH_PATH, index=False)
    save_json(best_lgbm_params, LGBM_PARAMS_PATH)
    print(lgbm_search_df)

    print("Training final LightGBM with best candidate")
    lgbm_model = lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        max_depth=-1,
        subsample_freq=1,
        scale_pos_weight=float(scale_pos_weight),
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        **best_lgbm_params,
    )

    lgbm_model.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        eval_names=["train", "valid"],
        eval_metric="average_precision",
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(200, first_metric_only=True), lgb.log_evaluation(100)],
    )

    lgbm_eval_history = lgbm_model.evals_result_

    joblib.dump(lgbm_model, LGBM_MODEL_PATH)
    save_json(lgbm_eval_history, LGBM_EVAL_PATH)

LightGBM candidate 1/5
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[3999]	valid_0's average_precision: 0.647691	valid_0's binary_logloss: 0.125285
Evaluated only: average_precision
LightGBM candidate 2/5
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's average_precision: 0.626015	valid_0's binary_logloss: 0.11554
Evaluated only: average_precision
LightGBM candidate 3/5
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[2490]	valid_0's average_precision: 0.650187	valid_0's binary_logloss: 0.130016
Evaluated only: average_precision
LightGBM candidate 4/5
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[4187]	valid_0's average_precision: 0.644693	valid_0's binary_logloss: 0.115202
Evaluated only: average_precision
LightGBM candidate 5/5
Tra

## Cell 13: Train XGBoost

This cell also uses a readable candidate search before training the final model.

In [15]:
XGB_MODEL_PATH = PHASE_ROOT / "xgb_model.pkl"
XGB_EVAL_PATH = PHASE_ROOT / "xgb_eval_history.json"
XGB_SEARCH_PATH = PHASE_ROOT / "xgb_search_results.csv"
XGB_PARAMS_PATH = PHASE_ROOT / "xgb_best_params.json"

if XGB_MODEL_PATH.exists() and not REFRESH_MODELS:
    print("Loading cached XGBoost model")
    xgb_model = joblib.load(XGB_MODEL_PATH)
    xgb_eval_history = json.load(open(XGB_EVAL_PATH, "r", encoding="utf-8"))
    best_xgb_params = load_json(XGB_PARAMS_PATH)
else:
    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_columns)
    dvalid = xgb.DMatrix(X_valid, label=y_valid, feature_names=feature_columns)

    xgb_candidates = [
        {
            "eta": 0.03,
            "max_depth": 8,
            "min_child_weight": 4,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "lambda": 2.0,
            "alpha": 0.0,
        },
        {
            "eta": 0.02,
            "max_depth": 10,
            "min_child_weight": 3,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "lambda": 3.0,
            "alpha": 0.0,
        },
        {
            "eta": 0.05,
            "max_depth": 6,
            "min_child_weight": 6,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "lambda": 2.0,
            "alpha": 0.0,
        },
        {
            "eta": 0.03,
            "max_depth": 7,
            "min_child_weight": 2,
            "subsample": 0.8,
            "colsample_bytree": 0.7,
            "lambda": 4.0,
            "alpha": 1.0,
        },
        {
            "eta": 0.02,
            "max_depth": 9,
            "min_child_weight": 8,
            "subsample": 0.75,
            "colsample_bytree": 0.75,
            "lambda": 5.0,
            "alpha": 0.5,
        },
    ] if RUN_MODEL_SEARCH else [
        {
            "eta": 0.03,
            "max_depth": 8,
            "min_child_weight": 4,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "lambda": 2.0,
            "alpha": 0.0,
        }
    ]

    xgb_search_rows = []
    best_xgb_score = -1.0
    best_xgb_params = None

    for idx, candidate in enumerate(xgb_candidates, start=1):
        print(f"XGBoost candidate {idx}/{len(xgb_candidates)}")

        candidate_params = {
            "objective": "binary:logistic",
            "eval_metric": "aucpr",
            "tree_method": "hist",
            "scale_pos_weight": float(scale_pos_weight),
            "seed": SEED,
            **candidate,
        }

        candidate_history = {}
        candidate_model = xgb.train(
            params=candidate_params,
            dtrain=dtrain,
            num_boost_round=4000,
            evals=[(dvalid, "valid")],
            evals_result=candidate_history,
            early_stopping_rounds=200,
            verbose_eval=False,
        )

        candidate_pred = candidate_model.predict(
            dvalid,
            iteration_range=(0, best_iteration_for_xgb(candidate_model)),
        )
        candidate_ap = average_precision_score(y_valid, candidate_pred)

        xgb_search_rows.append(
            {
                "candidate_id": idx,
                "valid_auc_pr": float(candidate_ap),
                "best_iteration": int(best_iteration_for_xgb(candidate_model)),
                **candidate,
            }
        )

        if candidate_ap > best_xgb_score:
            best_xgb_score = float(candidate_ap)
            best_xgb_params = candidate_params.copy()

    xgb_search_df = pd.DataFrame(xgb_search_rows).sort_values("valid_auc_pr", ascending=False)
    xgb_search_df.to_csv(XGB_SEARCH_PATH, index=False)
    save_json(best_xgb_params, XGB_PARAMS_PATH)
    print(xgb_search_df)

    xgb_eval_history = {}
    xgb_model = xgb.train(
        params=best_xgb_params,
        dtrain=dtrain,
        num_boost_round=4000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        evals_result=xgb_eval_history,
        early_stopping_rounds=200,
        verbose_eval=100,
    )

    joblib.dump(xgb_model, XGB_MODEL_PATH)
    save_json(xgb_eval_history, XGB_EVAL_PATH)

XGBoost candidate 1/5
XGBoost candidate 2/5
XGBoost candidate 3/5
XGBoost candidate 4/5
XGBoost candidate 5/5
   candidate_id  valid_auc_pr  best_iteration   eta  max_depth  \
1             2      0.646343            4000  0.02         10   
4             5      0.639072            3996  0.02          9   
0             1      0.623629            3998  0.03          8   
3             4      0.583195            1709  0.03          7   
2             3      0.543257             833  0.05          6   

   min_child_weight  subsample  colsample_bytree  lambda  alpha  
1                 3       0.80              0.80     3.0    0.0  
4                 8       0.75              0.75     5.0    0.5  
0                 4       0.85              0.85     2.0    0.0  
3                 2       0.80              0.70     4.0    1.0  
2                 6       0.90              0.80     2.0    0.0  
[0]	train-aucpr:0.46229	valid-aucpr:0.29142
[100]	train-aucpr:0.73373	valid-aucpr:0.44916
[200]	t

## Cell 14: Inference

In [16]:
valid_pred_lgbm = lgbm_model.predict_proba(X_valid)[:, 1]
test_pred_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

dvalid = xgb.DMatrix(X_valid, feature_names=feature_columns)
dtest = xgb.DMatrix(X_test, feature_names=feature_columns)

valid_pred_xgb = xgb_model.predict(dvalid, iteration_range=(0, xgb_model.best_iteration + 1))
test_pred_xgb = xgb_model.predict(dtest, iteration_range=(0, xgb_model.best_iteration + 1))

## Cell 14B: Benchmark Inference Latency
#
Proposal Table 1.1 expects a latency column, so we benchmark inference on the
frozen test matrix and report average milliseconds per scored transaction.

In [17]:
# %% [markdown]
# ## Cell 14B: Benchmark Inference Latency
#
# Benchmark both models from the same raw tabular input so the comparison is fair.
# For XGBoost, DMatrix creation is included inside the timed call.

# %%
LATENCY_PATH = PHASE_ROOT / "inference_latency_benchmarks.json"

latency_benchmark_df = X_test.sample(
    n=min(20000, len(X_test)),
    random_state=SEED,
).copy()

def predict_lgbm_from_frame(frame):
    return lgbm_model.predict_proba(frame)[:, 1]

def predict_xgb_from_frame(frame):
    dmatrix = xgb.DMatrix(frame, feature_names=feature_columns)
    return xgb_model.predict(
        dmatrix,
        iteration_range=(0, xgb_model.best_iteration + 1),
    )
print('hi1')
lgbm_latency = benchmark_latency_ms_per_row(
    predict_fn=predict_lgbm_from_frame,
    data=latency_benchmark_df,
)
print('hi2')
xgb_latency = benchmark_latency_ms_per_row(
    predict_fn=predict_xgb_from_frame,
    data=latency_benchmark_df,
)
print('hi1')
latency_metrics = {
    "LightGBM": lgbm_latency,
    "XGBoost": xgb_latency,
}

save_json(latency_metrics, LATENCY_PATH)
latency_metrics


hi1
hi2
hi1


{'LightGBM': {'mean_ms_per_row': 0.35628067134498903,
  'median_ms_per_row': 0.3555740914249782,
  'std_ms_per_row': 0.002853676905906326,
  'repeats': 20,
  'rows_benchmarked': 20000},
 'XGBoost': {'mean_ms_per_row': 0.06824055084751307,
  'median_ms_per_row': 0.06806331652501285,
  'std_ms_per_row': 0.0009840721389974124,
  'repeats': 20,
  'rows_benchmarked': 20000}}

## Cell 15: Save Predictions

In [18]:
PREDICTIONS_PATH = PHASE_ROOT / "baseline_predictions.parquet"

prediction_df = pd.concat(
    [
        pd.DataFrame(
            {
                "split": "valid",
                ID_COL: valid_df[ID_COL].values,
                TIME_COL: valid_df[TIME_COL].values,
                "y_true": y_valid,
                "lgbm_score": valid_pred_lgbm,
                "xgb_score": valid_pred_xgb,
            }
        ),
        pd.DataFrame(
            {
                "split": "test",
                ID_COL: test_df[ID_COL].values,
                TIME_COL: test_df[TIME_COL].values,
                "y_true": y_test,
                "lgbm_score": test_pred_lgbm,
                "xgb_score": test_pred_xgb,
            }
        ),
    ],
    ignore_index=True,
)

prediction_df.to_parquet(PREDICTIONS_PATH, index=False)
prediction_df.head()

,split,TransactionID,TransactionDT,y_true,lgbm_score,xgb_score
0,valid,3429905,11246665,0,4.820827e-07,0.002185
1,valid,3429906,11246704,0,6.873769e-09,0.000002
2,valid,3429908,11246761,0,2.570300e-06,0.000125
3,valid,3429907,11246761,0,4.120417e-07,0.000112
4,valid,3429909,11247072,0,1.906040e-06,0.000692


## Cell 16: Metrics

In [19]:
metrics = {}

metrics["valid_lgbm_auc_pr"] = float(average_precision_score(y_valid, valid_pred_lgbm))
metrics["valid_lgbm_auc_roc"] = float(roc_auc_score(y_valid, valid_pred_lgbm))
metrics["valid_lgbm_recall_at_1pct_fpr"] = recall_at_fpr(y_valid, valid_pred_lgbm)

metrics["valid_xgb_auc_pr"] = float(average_precision_score(y_valid, valid_pred_xgb))
metrics["valid_xgb_auc_roc"] = float(roc_auc_score(y_valid, valid_pred_xgb))
metrics["valid_xgb_recall_at_1pct_fpr"] = recall_at_fpr(y_valid, valid_pred_xgb)

metrics["test_lgbm_auc_pr"] = float(average_precision_score(y_test, test_pred_lgbm))
metrics["test_lgbm_auc_roc"] = float(roc_auc_score(y_test, test_pred_lgbm))
metrics["test_lgbm_recall_at_1pct_fpr"] = recall_at_fpr(y_test, test_pred_lgbm)

metrics["test_xgb_auc_pr"] = float(average_precision_score(y_test, test_pred_xgb))
metrics["test_xgb_auc_roc"] = float(roc_auc_score(y_test, test_pred_xgb))
metrics["test_xgb_recall_at_1pct_fpr"] = recall_at_fpr(y_test, test_pred_xgb)
metrics["test_lgbm_latency_ms"] = float(lgbm_latency["mean_ms_per_row"])
metrics["test_xgb_latency_ms"] = float(xgb_latency["mean_ms_per_row"])

for budget in ANALYST_BUDGETS:
    metrics[f"test_lgbm_precision_at_{budget}"] = precision_at_k(y_test, test_pred_lgbm, budget)
    metrics[f"test_lgbm_recall_at_{budget}"] = recall_at_k(y_test, test_pred_lgbm, budget)
    metrics[f"test_xgb_precision_at_{budget}"] = precision_at_k(y_test, test_pred_xgb, budget)
    metrics[f"test_xgb_recall_at_{budget}"] = recall_at_k(y_test, test_pred_xgb, budget)

METRICS_PATH = PHASE_ROOT / "baseline_metrics.json"
save_json(metrics, METRICS_PATH)

metrics

{'valid_lgbm_auc_pr': 0.6501874726370235,
 'valid_lgbm_auc_roc': 0.9255135110524025,
 'valid_lgbm_recall_at_1pct_fpr': 0.5999008428358948,
 'valid_xgb_auc_pr': 0.6463429755178337,
 'valid_xgb_auc_roc': 0.9271319752960854,
 'valid_xgb_recall_at_1pct_fpr': 0.5899851264253843,
 'test_lgbm_auc_pr': 0.5806981612472414,
 'test_lgbm_auc_roc': 0.8997861962571055,
 'test_lgbm_recall_at_1pct_fpr': 0.5231916963996107,
 'test_xgb_auc_pr': 0.5867185540140676,
 'test_xgb_auc_roc': 0.9137043944107601,
 'test_xgb_recall_at_1pct_fpr': 0.5157314304249108,
 'test_lgbm_latency_ms': 0.35628067134498903,
 'test_xgb_latency_ms': 0.06824055084751307,
 'test_lgbm_precision_at_100': 1.0,
 'test_lgbm_recall_at_100': 0.032435939020434644,
 'test_xgb_precision_at_100': 1.0,
 'test_xgb_recall_at_100': 0.032435939020434644,
 'test_lgbm_precision_at_250': 0.984,
 'test_lgbm_recall_at_250': 0.07979240999026922,
 'test_xgb_precision_at_250': 0.944,
 'test_xgb_recall_at_250': 0.07654881608822575,
 'test_lgbm_precision_a

## Cell 16B: Temporal OOF Predictions For Meta-Learning
#
Publication-oriented Phase 3 needs more supervision than the single validation
split can provide. We therefore generate leakage-safe temporal OOF predictions
inside the training period only.

In [20]:
OOF_PREDICTIONS_PATH = PHASE_ROOT / "baseline_oof_predictions.parquet"
OOF_SUMMARY_PATH = PHASE_ROOT / "baseline_oof_summary.json"
OOF_FOLDS_PATH = PHASE_ROOT / "temporal_oof_folds.json"

oof_folds = build_temporal_oof_folds(len(train_df))

if OOF_PREDICTIONS_PATH.exists() and OOF_SUMMARY_PATH.exists() and OOF_FOLDS_PATH.exists() and not REFRESH_MODELS:
    baseline_oof_predictions = pd.read_parquet(OOF_PREDICTIONS_PATH)
    baseline_oof_summary = load_json(OOF_SUMMARY_PATH)
else:
    oof_rows = []
    fold_summary_rows = []

    for fold in oof_folds:
        fold_id = fold["fold_id"]
        train_end = fold["train_end"]
        eval_start = fold["eval_start"]
        eval_end = fold["eval_end"]

        print('im')

        print(
            f"OOF fold {fold_id}: train [0:{train_end}) -> predict [{eval_start}:{eval_end})"
        )

        X_prefix = X_train.iloc[:train_end].copy()
        y_prefix = y_train[:train_end]
        X_eval = X_train.iloc[eval_start:eval_end].copy()
        y_eval = y_train[eval_start:eval_end]
        eval_frame = train_df.iloc[eval_start:eval_end].copy()

        lgbm_fold_model, lgbm_fold_best_iter = fit_lgbm_with_inner_early_stopping(
            X_prefix=X_prefix,
            y_prefix=y_prefix,
            categorical_columns=categorical_columns,
            best_params=best_lgbm_params,
        )
        lgbm_oof_score = lgbm_fold_model.predict_proba(X_eval)[:, 1]

        xgb_fold_model, xgb_fold_best_iter = fit_xgb_with_inner_early_stopping(
            X_prefix=X_prefix,
            y_prefix=y_prefix,
            feature_columns=feature_columns,
            best_params=best_xgb_params,
        )
        deval_fold = xgb.DMatrix(X_eval, feature_names=feature_columns)
        xgb_oof_score = xgb_fold_model.predict(deval_fold, iteration_range=(0, xgb_fold_best_iter))

        fold_ap_lgbm = float(average_precision_score(y_eval, lgbm_oof_score))
        fold_ap_xgb = float(average_precision_score(y_eval, xgb_oof_score))
        fold_summary_rows.append(
            {
                "fold_id": int(fold_id),
                "train_end": int(train_end),
                "eval_start": int(eval_start),
                "eval_end": int(eval_end),
                "rows_predicted": int(eval_end - eval_start),
                "lgbm_auc_pr": fold_ap_lgbm,
                "xgb_auc_pr": fold_ap_xgb,
                "lgbm_best_iteration": int(lgbm_fold_best_iter),
                "xgb_best_iteration": int(xgb_fold_best_iter),
            }
        )

        oof_rows.append(
            pd.DataFrame(
                {
                    "fold_id": int(fold_id),
                    "split": "train_oof",
                    ID_COL: eval_frame[ID_COL].to_numpy(),
                    TIME_COL: eval_frame[TIME_COL].to_numpy(),
                    "y_true": y_eval,
                    "lgbm_score": lgbm_oof_score,
                    "xgb_score": xgb_oof_score,
                }
            )
        )

    baseline_oof_predictions = pd.concat(oof_rows, axis=0, ignore_index=True).sort_values(TIME_COL).reset_index(drop=True)
    baseline_oof_summary = {
        "warmup_fraction": OOF_WARMUP_FRACTION,
        "num_folds": OOF_NUM_FOLDS,
        "rows_covered": int(len(baseline_oof_predictions)),
        "coverage_fraction_of_train": float(len(baseline_oof_predictions) / max(len(train_df), 1)),
        "folds": fold_summary_rows,
        "overall_lgbm_auc_pr": float(average_precision_score(baseline_oof_predictions["y_true"], baseline_oof_predictions["lgbm_score"])),
        "overall_xgb_auc_pr": float(average_precision_score(baseline_oof_predictions["y_true"], baseline_oof_predictions["xgb_score"])),
    }

    baseline_oof_predictions.to_parquet(OOF_PREDICTIONS_PATH, index=False)
    save_json(baseline_oof_summary, OOF_SUMMARY_PATH)
    save_json(oof_folds, OOF_FOLDS_PATH)
    print('in2')

baseline_oof_summary

im
OOF fold 1: train [0:177162) -> predict [177162:265743)
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[2500]	valid_0's average_precision: 0.750358	valid_0's binary_logloss: 0.128391
Evaluated only: average_precision
im
OOF fold 2: train [0:265743) -> predict [265743:354324)
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[2498]	valid_0's average_precision: 0.756759	valid_0's binary_logloss: 0.13217
Evaluated only: average_precision
im
OOF fold 3: train [0:354324) -> predict [354324:442905)
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[2490]	valid_0's average_precision: 0.583017	valid_0's binary_logloss: 0.148838
Evaluated only: average_precision
in2


{'warmup_fraction': 0.4,
 'num_folds': 3,
 'rows_covered': 265743,
 'coverage_fraction_of_train': 0.6,
 'folds': [{'fold_id': 1,
   'train_end': 177162,
   'eval_start': 177162,
   'eval_end': 265743,
   'rows_predicted': 88581,
   'lgbm_auc_pr': 0.6209318932337096,
   'xgb_auc_pr': 0.6324630393556757,
   'lgbm_best_iteration': 2500,
   'xgb_best_iteration': 3999},
  {'fold_id': 2,
   'train_end': 265743,
   'eval_start': 265743,
   'eval_end': 354324,
   'rows_predicted': 88581,
   'lgbm_auc_pr': 0.5891006639802611,
   'xgb_auc_pr': 0.5989721625795796,
   'lgbm_best_iteration': 2498,
   'xgb_best_iteration': 3981},
  {'fold_id': 3,
   'train_end': 354324,
   'eval_start': 354324,
   'eval_end': 442905,
   'rows_predicted': 88581,
   'lgbm_auc_pr': 0.6609691593859351,
   'xgb_auc_pr': 0.6740061043895659,
   'lgbm_best_iteration': 2490,
   'xgb_best_iteration': 3996}],
 'overall_lgbm_auc_pr': 0.624520956364999,
 'overall_xgb_auc_pr': 0.6354230341263686}

## Cell 17: Save Raw RQ1 Support Exports
#
This phase now saves only raw scientific outputs.
Proposal-facing RQ1 figures and tables should be generated separately by
`rq1_artifacts.py`.

In [21]:
baseline_summary_raw = pd.DataFrame(
    [
        {
            "Model": "LightGBM",
            "AUC-ROC": metrics["test_lgbm_auc_roc"],
            "AUC-PR": metrics["test_lgbm_auc_pr"],
            "Recall@1%FPR": metrics["test_lgbm_recall_at_1pct_fpr"],
            "Latency(ms)": metrics["test_lgbm_latency_ms"],
        },
        {
            "Model": "XGBoost",
            "AUC-ROC": metrics["test_xgb_auc_roc"],
            "AUC-PR": metrics["test_xgb_auc_pr"],
            "Recall@1%FPR": metrics["test_xgb_recall_at_1pct_fpr"],
            "Latency(ms)": metrics["test_xgb_latency_ms"],
        },
    ]
)
baseline_summary_raw.to_csv(PHASE_ROOT / "baseline_summary_raw.csv", index=False)

precision_lgbm, recall_lgbm, _ = precision_recall_curve(y_test, test_pred_lgbm)
precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, test_pred_xgb)
pr_curve_raw = pd.concat(
    [
        pd.DataFrame({"Model": "LightGBM", "Recall": recall_lgbm, "Precision": precision_lgbm}),
        pd.DataFrame({"Model": "XGBoost", "Recall": recall_xgb, "Precision": precision_xgb}),
    ],
    ignore_index=True,
)
pr_curve_raw.to_csv(PHASE_ROOT / "precision_recall_curve_points.csv", index=False)

budget_rows = []
for budget in ANALYST_BUDGETS:
    budget_rows.append({"Budget": budget, "Model": "LightGBM", "Recall": metrics[f"test_lgbm_recall_at_{budget}"]})
    budget_rows.append({"Budget": budget, "Model": "XGBoost", "Recall": metrics[f"test_xgb_recall_at_{budget}"]})

budget_df = pd.DataFrame(budget_rows)
budget_df.to_csv(PHASE_ROOT / "recall_vs_budget_raw.csv", index=False)
save_json(lgbm_eval_history, PHASE_ROOT / "lgbm_eval_history_for_reporting.json")
save_json(xgb_eval_history, PHASE_ROOT / "xgb_eval_history_for_reporting.json")

## Cell 18: Feature Importance Exports

In [22]:
lgbm_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance_gain": lgbm_model.booster_.feature_importance(importance_type="gain"),
        "importance_split": lgbm_model.booster_.feature_importance(importance_type="split"),
    }
).sort_values("importance_gain", ascending=False)

xgb_gain_map = xgb_model.get_score(importance_type="gain")
xgb_weight_map = xgb_model.get_score(importance_type="weight")
xgb_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance_gain": [xgb_gain_map.get(col, 0.0) for col in feature_columns],
        "importance_weight": [xgb_weight_map.get(col, 0.0) for col in feature_columns],
    }
).sort_values("importance_gain", ascending=False)

lgbm_importance.to_csv(PHASE_ROOT / "lgbm_feature_importance.csv", index=False)
xgb_importance.to_csv(PHASE_ROOT / "xgb_feature_importance.csv", index=False)

lgbm_importance.head(10), xgb_importance.head(10)

(             feature  importance_gain  importance_split
 441    card_addr_key     1.417911e+06             21965
 443   card_email_key     1.408407e+06             24308
 442       card12_key     1.013824e+06             13612
 310             V258     8.080740e+05               536
 444  card_device_key     7.907567e+05             15884
 346             V294     4.441505e+05               904
 28               C14     3.983051e+05              2572
 122              V70     2.689157e+05               453
 30                D2     1.915040e+05              5583
 18                C4     1.410406e+05               756,
     feature  importance_gain  importance_weight
 310    V258      3884.974121              580.0
 122     V70      2025.223755              364.0
 309    V257       800.012390              198.0
 143     V91       732.304321              306.0
 346    V294       676.195862              943.0
 253    V201       470.195038              165.0
 347    V295       213.990189

## Cell 19: Dataset Statistics Table

In [23]:
dataset_table = pd.DataFrame(
    [
        {
            "Split": "Train",
            "Rows": len(train_df),
            "FraudRate": float(train_df[TARGET_COL].mean()),
            "TimeMin": int(train_df[TIME_COL].min()),
            "TimeMax": int(train_df[TIME_COL].max()),
        },
        {
            "Split": "Valid",
            "Rows": len(valid_df),
            "FraudRate": float(valid_df[TARGET_COL].mean()),
            "TimeMin": int(valid_df[TIME_COL].min()),
            "TimeMax": int(valid_df[TIME_COL].max()),
        },
        {
            "Split": "Test",
            "Rows": len(test_df),
            "FraudRate": float(test_df[TARGET_COL].mean()),
            "TimeMin": int(test_df[TIME_COL].min()),
            "TimeMax": int(test_df[TIME_COL].max()),
        },
    ]
)

dataset_table.to_csv(PHASE_ROOT / "dataset_statistics.csv", index=False)
dataset_table

,Split,Rows,FraudRate,TimeMin,TimeMax
0,Train,442905,0.035138,86400,11246605
1,Valid,59054,0.034155,11246665,13151840
2,Test,88581,0.034804,13151880,15811131


## Cell 20: Save Manifest

In [24]:
manifest = {
    "phase": "phase01_data_baselines",
    "files": {
        "train": str(TRAIN_PATH),
        "valid": str(VALID_PATH),
        "test": str(TEST_PATH),
        "feature_schema": str(FEATURE_SCHEMA_PATH),
        "category_maps": str(CATEGORY_MAPS_PATH),
        "split_info": str(SPLIT_INFO_PATH),
        "lgbm_model": str(LGBM_MODEL_PATH),
        "lgbm_best_params": str(LGBM_PARAMS_PATH),
        "lgbm_search_results": str(LGBM_SEARCH_PATH),
        "xgb_model": str(XGB_MODEL_PATH),
        "xgb_best_params": str(XGB_PARAMS_PATH),
        "xgb_search_results": str(XGB_SEARCH_PATH),
        "predictions": str(PREDICTIONS_PATH),
        "baseline_oof_predictions": str(OOF_PREDICTIONS_PATH),
        "baseline_oof_summary": str(OOF_SUMMARY_PATH),
        "temporal_oof_folds": str(OOF_FOLDS_PATH),
        "metrics": str(METRICS_PATH),
        "latency_benchmarks": str(LATENCY_PATH),
        "baseline_summary_raw": str(PHASE_ROOT / "baseline_summary_raw.csv"),
        "precision_recall_curve_points": str(PHASE_ROOT / "precision_recall_curve_points.csv"),
        "recall_vs_budget_raw": str(PHASE_ROOT / "recall_vs_budget_raw.csv"),
        "lgbm_eval_history_for_reporting": str(PHASE_ROOT / "lgbm_eval_history_for_reporting.json"),
        "xgb_eval_history_for_reporting": str(PHASE_ROOT / "xgb_eval_history_for_reporting.json"),
        "lgbm_feature_importance": str(PHASE_ROOT / "lgbm_feature_importance.csv"),
        "xgb_feature_importance": str(PHASE_ROOT / "xgb_feature_importance.csv"),
        "dataset_statistics": str(PHASE_ROOT / "dataset_statistics.csv"),
    },
}

save_json(manifest, OUTPUT_ROOT / "manifests" / "phase01_artifact_manifest.json")
manifest

{'phase': 'phase01_data_baselines',
 'files': {'train': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/train.parquet',
  'valid': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/valid.parquet',
  'test': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/test.parquet',
  'feature_schema': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/feature_schema.json',
  'category_maps': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/category_maps.json',
  'split_info': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/split_info.json',
  'lgbm_model': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/lgbm_model.pkl',
  'lgbm_best_params': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/lgbm_best_params.json',
  'lgbm_search_results': '/kaggle/working/thesis_outputs/artifacts/phase01_data_baselines/lgbm_search_results.csv',
  'xgb_model': '/kaggle/working/thesis_output

## Cell 21: Hand-Off To Notebook 02
#
Notebook 02 should read:
- `train.parquet`
- `valid.parquet`
- `test.parquet`
- `feature_schema.json`
- `baseline_predictions.parquet`

In [25]:
gc.collect()
print("Phase 01 finished.")
print("Phase 01 now saves raw scientific outputs only.")
print("Generate proposal-facing RQ1 artifacts separately with rq1_artifacts.py.")

Phase 01 finished.
Phase 01 now saves raw scientific outputs only.
Generate proposal-facing RQ1 artifacts separately with rq1_artifacts.py.
